Ноутбук считает метрики для E на русском объединенном корпусе. Эта оценка нужна для сравнения с BM, исходным PromptRetriever и дообученным LLaMA-ретривером.


In [ ]:
import os, json, math, hashlib, time
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
import faiss
from tqdm.auto import tqdm


ENC_DIR = "/kaggle/working/e5_proxy_encoding"
METRIC_TESTSET = "/kaggle/input/datasets/sukiss/dataset-for-metrics/chunks_testset_metric_instructions.jsonl"

CORPUS_MODE = "combined"

OUT_EVAL_DIR = "/kaggle/working/e5_metric_eval"
os.makedirs(OUT_EVAL_DIR, exist_ok=True)

PER_QUERY_OUT = os.path.join(OUT_EVAL_DIR, f"e5_metric_eval_{CORPUS_MODE}_per_query.jsonl")
SUMMARY_OUT = os.path.join(OUT_EVAL_DIR, f"e5_metric_eval_{CORPUS_MODE}_summary.json")

QUERY_MAX_LEN = 192
BATCH_QUERIES = 64


def sha1_text(t):
    return hashlib.sha1(t.strip().encode("utf-8")).hexdigest()

def load_docids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]

def infer_dim_from_file(path, n_rows, dtype=np.float16):
    return os.path.getsize(path) // (n_rows * np.dtype(dtype).itemsize)

def load_one_corpus(enc_dir, name):
    corpus_path = os.path.join(enc_dir, f"{name}_corpus.jsonl")
    docids_path = os.path.join(enc_dir, f"{name}_docids.txt")
    emb_path = os.path.join(enc_dir, f"{name}_embeddings.npy")

    docids = load_docids(docids_path)
    dim = infer_dim_from_file(emb_path, len(docids), dtype=np.float16)
    emb = np.memmap(emb_path, dtype=np.float16, mode="r", shape=(len(docids), dim))

    hashes = []
    with open(corpus_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            hashes.append(sha1_text(obj["text"]))

    return docids, hashes, emb, dim

def load_corpus(enc_dir, mode):
    if mode == "test":
        docids, hashes, emb, dim = load_one_corpus(enc_dir, "test")
        return [f"test::{d}" for d in docids], hashes, np.asarray(emb, dtype=np.float32), dim

    if mode == "main":
        docids, hashes, emb, dim = load_one_corpus(enc_dir, "main")
        return [f"main::{d}" for d in docids], hashes, np.asarray(emb, dtype=np.float32), dim

    if mode == "combined":
        main_docids, main_hashes, main_emb, dim1 = load_one_corpus(enc_dir, "main")
        test_docids, test_hashes, test_emb, dim2 = load_one_corpus(enc_dir, "test")
        assert dim1 == dim2

        docids = [f"main::{d}" for d in main_docids] + [f"test::{d}" for d in test_docids]
        hashes = main_hashes + test_hashes

        xb = np.vstack([
            np.asarray(main_emb, dtype=np.float32),
            np.asarray(test_emb, dtype=np.float32),
        ])

        return docids, hashes, xb, dim1

    raise ValueError(mode)

def load_metric_items(path):
    items = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            it = json.loads(line)

            if all(it.get(k) for k in [
                "query_id",
                "query",
                "only_instruction",
                "reverse_instruction",
                "pmrr_instruction",
                "positive",
                "pmrr_changed_docs",
            ]):
                items.append(it)

    return items

def query_text(it, mode):
    q = it["query"].strip()

    if mode == "orig":
        return q
    if mode == "inst":
        return (q + " " + it["only_instruction"].strip()).strip()
    if mode == "rev":
        return (q + " " + it["reverse_instruction"].strip()).strip()
    if mode == "pmrr":
        return (q + " " + it["pmrr_instruction"].strip()).strip()

    raise ValueError(mode)

def mrr_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def ndcg_at_k(rank, k):
    return 1.0 / math.log2(rank + 1) if rank <= k else 0.0

def ap_at_k(rank, k):
    return 1.0 / rank if rank <= k else 0.0

def hit_at_k(rank, k):
    return 1.0 if rank <= k else 0.0

def sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev):
    return float(
        (r_ins < r_ori) and
        (s_ins > s_ori) and
        (r_ori < r_rev) and
        (s_ori > s_rev)
    )

def wise_score(r_ori, r_ins, r_rev, n_positive_original=1, k=20):
    if r_ins <= r_ori < r_rev:
        if r_ori <= n_positive_original and r_ins == 1:
            return 1.0
        if r_ori <= k:
            return (1.0 - (math.sqrt(max(0, r_ori - r_ins)) / k)) * (1.0 / math.sqrt(r_ins))
        return 0.01

    if r_rev < r_ori < r_ins:
        return -1.0
    if r_ori <= r_ins:
        return (r_ori - r_ins) / r_ins
    if r_rev <= r_ori:
        return (r_rev - r_ori) / r_ori

    return 0.0

def pmrr_doc_score(rank_old, rank_new):
    rr_old = 1.0 / rank_old
    rr_new = 1.0 / rank_new

    if rank_old > rank_new:
        return (rr_old / rr_new) - 1.0
    return 1.0 - (rr_new / rr_old)

def best_rank_score(I_row, D_row, candidate_indices):
    candidates = set(candidate_indices)

    for pos, idx in enumerate(I_row):
        if int(idx) in candidates:
            return pos + 1, float(D_row[pos])

    return len(I_row) + 1, float("-inf")

def summarize_mode(rows, mode):
    ranks = [r[f"rank_{mode}"] for r in rows]

    return {
        "MRR@10": float(np.mean([mrr_at_k(r, 10) for r in ranks])),
        "nDCG@5": float(np.mean([ndcg_at_k(r, 5) for r in ranks])),
        "nDCG@10": float(np.mean([ndcg_at_k(r, 10) for r in ranks])),
        "MAP@1000": float(np.mean([ap_at_k(r, 1000) for r in ranks])),
        "Hit@10": float(np.mean([hit_at_k(r, 10) for r in ranks])),
    }

def summarize_rows(rows):
    pmrr_values = [r["pmrr"] for r in rows if r["pmrr"] is not None]

    return {
        "count": len(rows),
        "orig": summarize_mode(rows, "orig"),
        "inst": summarize_mode(rows, "inst"),
        "rev": summarize_mode(rows, "rev"),
        "instruction_metrics": {
            "SICR": float(np.mean([r["sicr"] for r in rows])),
            "SICR_x100": float(100 * np.mean([r["sicr"] for r in rows])),
            "WISE": float(np.mean([r["wise"] for r in rows])),
            "WISE_x100": float(100 * np.mean([r["wise"] for r in rows])),
            "pMRR": float(np.mean(pmrr_values)) if pmrr_values else None,
            "pMRR_x100": float(100 * np.mean(pmrr_values)) if pmrr_values else None,
        },
    }


docids, hashes, xb, dim = load_corpus(ENC_DIR, CORPUS_MODE)

hash_to_indices = defaultdict(list)
for i, h in enumerate(hashes):
    hash_to_indices[h].append(i)

index = faiss.IndexFlatIP(dim)
index.add(xb)

print("Corpus mode:", CORPUS_MODE)
print("Docs:", len(docids), "dim:", dim)


items = load_metric_items(METRIC_TESTSET)

kept = []
missing_positive = 0

for it in items:
    if sha1_text(it["positive"]) not in hash_to_indices:
        missing_positive += 1
        continue
    kept.append(it)

items = kept

print("Metric items kept:", len(items))
print("Missing positive:", missing_positive)

@torch.inference_mode()
def encode_queries(texts):
    all_emb = []

    for i in tqdm(range(0, len(texts), BATCH_QUERIES), desc="encoding queries"):
        batch_texts = texts[i:i+BATCH_QUERIES]
        emb = encode_texts(batch_texts, QUERY_MAX_LEN, is_query=True)
        all_emb.append(emb.astype(np.float32))

    return np.vstack(all_emb)

q_orig = encode_queries([query_text(it, "orig") for it in items])
q_inst = encode_queries([query_text(it, "inst") for it in items])
q_rev = encode_queries([query_text(it, "rev") for it in items])
q_pmrr = encode_queries([query_text(it, "pmrr") for it in items])


topk = len(docids)

D_orig, I_orig = index.search(q_orig, topk)
D_inst, I_inst = index.search(q_inst, topk)
D_rev, I_rev = index.search(q_rev, topk)
D_pmrr, I_pmrr = index.search(q_pmrr, topk)


rows = []
stats = Counter()

for i, it in enumerate(items):
    pos_candidates = hash_to_indices[sha1_text(it["positive"])]

    r_ori, s_ori = best_rank_score(I_orig[i], D_orig[i], pos_candidates)
    r_ins, s_ins = best_rank_score(I_inst[i], D_inst[i], pos_candidates)
    r_rev, s_rev = best_rank_score(I_rev[i], D_rev[i], pos_candidates)

    changed_scores = []
    changed_details = []

    for ch in it.get("pmrr_changed_docs", []):
        h = ch.get("text_hash") or sha1_text(ch.get("text", ""))
        cand = hash_to_indices.get(h)

        if not cand:
            stats["missing_pmrr_changed_doc"] += 1
            continue

        r_old, s_old = best_rank_score(I_inst[i], D_inst[i], cand)
        r_new, s_new = best_rank_score(I_pmrr[i], D_pmrr[i], cand)

        score = pmrr_doc_score(r_old, r_new)
        changed_scores.append(score)

        changed_details.append({
            "doc_id": ch.get("doc_id"),
            "rank_old_inst": r_old,
            "score_old_inst": s_old,
            "rank_new_pmrr": r_new,
            "score_new_pmrr": s_new,
            "pmrr_doc_score": score,
            "reason": ch.get("reason", ""),
            "text_hash": h,
        })

    rows.append({
        "query_id": str(it["query_id"]),
        "instruction_style": it.get("instruction_style", "unknown"),

        "rank_orig": r_ori,
        "score_orig": s_ori,
        "rank_inst": r_ins,
        "score_inst": s_ins,
        "rank_rev": r_rev,
        "score_rev": s_rev,

        "sicr": sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev),
        "wise": wise_score(r_ori, r_ins, r_rev),
        "pmrr": float(np.mean(changed_scores)) if changed_scores else None,
        "pmrr_changed_docs_eval": changed_details,
    })

with open(PER_QUERY_OUT, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

summary = summarize_rows(rows)

by_style = {}
for style in sorted(set(r["instruction_style"] for r in rows)):
    style_rows = [r for r in rows if r["instruction_style"] == style]
    by_style[style] = summarize_rows(style_rows)

summary["model"] = "E5-multilingual-base fine-tuned"
summary["corpus_mode"] = CORPUS_MODE
summary["docs"] = len(docids)
summary["by_style"] = by_style
summary["stats"] = dict(stats)
summary["outputs"] = {
    "per_query": PER_QUERY_OUT,
    "summary": SUMMARY_OUT,
}

with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDONE")
print("Per-query:", PER_QUERY_OUT)
print("Summary:", SUMMARY_OUT)

print(json.dumps({
    "model": summary["model"],
    "count": summary["count"],
    "orig": summary["orig"],
    "inst": summary["inst"],
    "rev": summary["rev"],
    "instruction_metrics": summary["instruction_metrics"],
    "stats": summary["stats"],
}, ensure_ascii=False, indent=2))

print("\nBY STYLE:")
for style, s in summary["by_style"].items():
    print(style, json.dumps({
        "count": s["count"],
        "SICR_x100": s["instruction_metrics"]["SICR_x100"],
        "WISE_x100": s["instruction_metrics"]["WISE_x100"],
        "pMRR_x100": s["instruction_metrics"]["pMRR_x100"],
    }, ensure_ascii=False))
